# Experimental training notebook
This notebook has not been validated on the current Colab GPU runtime. Its dependency pins and trainer API may require compatible versions. Read `docs/TRAINING.md` before running it. Use reviewed data with independent splits, and keep test examples out of training. No fine-tuned weights, speed measurements, or legal-compliance certification are supplied by this repository.


# 🇩🇪 Fine-Tuning Rechly-DE-3B with Unsloth

This notebook fine-tunes **Rechly-DE-3B** (based on `Qwen/Qwen2.5-3B-Instruct`) for 100% local, GoBD-compliant German invoice generation, expense categorization, and business correspondence.

⚡ **Hardware Requirement**: Google Colab Free Tier (T4 GPU with 15 GB VRAM is sufficient!). Training time is ~25-35 minutes for 1,500 samples.

In [ ]:
# Verify GPU allocation
!nvidia-smi

### 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

### 2. Load Base Model in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect: Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # 4bit quantization to drastically reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

### 3. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

### 4. Load Dataset & Apply ChatML Template

In [ ]:
from datasets import load_dataset
import os

# Upload your 'rechly_german_accounting.jsonl' or use a seed dataset
dataset_file = "rechly_german_accounting.jsonl"
if not os.path.exists(dataset_file):
    print(f"File {dataset_file} not found. Upload it using the Colab file browser!")
else:
    dataset = load_dataset("json", data_files=dataset_file, split="train")

    def formatting_prompts_func(examples):
        convs = examples["conversations"]
        texts = []
        for conv in convs:
            formatted = []
            for msg in conv:
                role = "user" if msg["from"] == "human" else "assistant"
                formatted.append({"role": role, "content": msg["value"]})
            text = tokenizer.apply_chat_template(formatted, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return {"text": texts}

    dataset = dataset.map(formatting_prompts_func, batched=True)
    print(f"Successfully loaded and formatted {len(dataset)} training samples.")

### 5. Train Model with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150, # or num_train_epochs=3
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="rechly_model_outputs",
    ),
)

trainer_stats = trainer.train()
print(f"Training finished with final loss: {trainer_stats.training_loss:.4f}")

### 6. Interactive Testing & Sanity Check

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Erstelle eine Rechnung an WebTech GmbH: 14 Stunden Frontend-Entwicklung à 95€ und 1x SSL Zertifikat Pauschal 60€, zahlbar in 14 Tagen mit 2% Skonto."}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.2)
response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Model Output:")
print(response)

### 7. Export Directly to GGUF for Ollama

In [ ]:
# Export quantized Q4_K_M GGUF
model.save_pretrained_gguf("rechly-de-3b", tokenizer, quantization_method="q4_k_m")
print("Export complete! Check your file browser for 'rechly-de-3b-q4_k_m.gguf'")